# 04 — Fine-tune on a single catalog window (pool12 slice)

Fine-tune a **pretrained** Shalow model ([`04_Sh_train_select_window_12.ipynb`](../04_Sh_train_select_window_12.ipynb)) on **one catalog window only** (default **W11**).

| Item | Value |
|---|---|
| **Change window** | Set `FINETUNE_WINDOW = 1..12` in setup |
| Pretrained weights | `Models/adv_no/..._select_window_12_l1random_slots6/best_model.pth` (from [`04_Sh_train_select_window_12.ipynb`](../04_Sh_train_select_window_12.ipynb)) |
| Train/val data | Existing **`select_window_12`** pool export — **no re-export** (slice catalog slot) |
| Epochs | 100 |
| Checkpoints | `Models/fineTuning/{RUN_NAME}/` |
| Eval | [`05_Sh_finetune_eval_rollout.ipynb`](05_Sh_finetune_eval_rollout.ipynb) |

**Re-export?** No — use `Shalow/data/select_window_12/sample_13_22_uv_dt360_sims_pool_k1.npy` and pick slot `FINETUNE_WINDOW - 1`.

**Run order:** setup → export check → train config → train → loss curves.


In [ ]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
# Notebook lives in Shalow/fineTuning/ — chdir to Shalow for imports/paths.
SHALOW_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "fineTuning" else (
    NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR / "Shalow"
)
ROOT = SHALOW_DIR.parent
os.chdir(SHALOW_DIR)
for _p in (ROOT, SHALOW_DIR, SHALOW_DIR / "fineTuning"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import run_registry as _run_registry
importlib.reload(_run_registry)

from window_paths import pool_prefix, test_prefix
from run_registry import (
    RUN_TAG,
    finetune_run_name,
    weight_decay_run_tag,
    generator_run_tag,
    dropout_run_tag,
)
from window_finetune_common import resolve_finetune_window, list_catalog_windows

import train.adv_no_train as _adv
importlib.reload(_adv)

from train.adv_no_train import AdvNOTrainConfig, run_adv_no_training

# ========== CHANGE THESE ==========
FINETUNE_WINDOW = 8  # user-facing catalog window 1..12------------------------------------------------>
# ==================================

win = resolve_finetune_window(FINETUNE_WINDOW)
WINDOW_SLOT = win["window_slot"]
CATALOG_EXPORT_KEY = win["catalog_export_key"]
CATALOG_LABEL = win["catalog_label"]

EXPORT_KEY = "select_window_12"
DATA_DIR = SHALOW_DIR / "data"
DT = 360
DATA_PREFIX = pool_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
TEST_DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)

# Pretrained model from 04_Sh_train_select_window_12.ipynb (pool12, k=6 slots/epoch)
PRETRAINED_RUN_NAME = (
    "shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6"
)
PRETRAINED_RUN_DIR = SHALOW_DIR / "Models" / "adv_no" / PRETRAINED_RUN_NAME
INIT_MODEL_PATH = PRETRAINED_RUN_DIR / "best_model.pth"

MODELS_DIR = SHALOW_DIR / "Models" / "fineTuning"
LOG_DIR = SHALOW_DIR / "logs" / "fineTuning"
for _d in (MODELS_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES = None
MAX_MASK_FRAC = 0.4  # match 01_export_select_window_pool12
VAL_SPLIT = 0.3
VAL_SPLIT_MODE = "random"
VAL_SPLIT_SEED = 42

USE_WINDOW_POOL = True
DYNAMIC_RANDOM_CROP = True  # required for window-pool path
RANDOM_CROP_SEED = 42
TRAIN_CROPS_PER_ANCHOR = 1
TRAIN_SLOTS_PER_EPOCH = 1
N_WINDOWS_PER_SAMPLE = 12

NUM_WORKERS = min(os.cpu_count() or 8, 8)
PIN_MEMORY = True
USE_MULTI_GPU = False
DEVICE = "cuda:1"   #---------------------------------------------------------------------->

GENERATOR = "gUnet"
UNET_DIM = 32
UNET_DIM_MULTS = (1, 2, 4, 8)
UNET_DROPOUT = 0.1

EPOCHS = 100
OPTIMIZER = "adamw"
WEIGHT_DECAY = 1e-3

LOSS_MODE = "l1"
LOSS_METRIC = "components"
LOSS_SPATIAL_MODE = "full"
CHANNEL_NAMES = ("u", "v")
iD = 2
BETA = 0.1
LAMBDA_PERC = 0.1
PERCEPTUAL_BACKEND = "vgg"
PERCEPTUAL_WEIGHTS = (0.1, 0.1, 1.0, 1.0, 1.0)
LPIPS_NET = "alex"
PERCEPTUAL_WARMUP_EPOCHS = 999
WARMUP_EPOCHS = 999
D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2
USE_COSINE_LR = True
VAL_RECONSTRUCTION_METRIC = "l1"

BASE_RUN_TAG = RUN_TAG
RUN_TAG = weight_decay_run_tag(BASE_RUN_TAG, WEIGHT_DECAY)
RUN_TAG = dropout_run_tag(RUN_TAG, UNET_DROPOUT)
RUN_TAG = generator_run_tag(RUN_TAG, GENERATOR)
RUN_NAME = finetune_run_name(
    EXPORT_KEY, FINETUNE_WINDOW, dt=DT, run_tag=RUN_TAG, loss_tag="l1random"
)
RUN_DIR = MODELS_DIR / RUN_NAME

DETACHED = True
RESUME = False
PROTECT_EXISTING_RUN = False
PREVIEW_LOSS_MASK = True
PREVIEW_VAL_MASK = True
PREVIEW_VAL_MASK_SAMPLES = 3
SHOW_INLINE_PLOTS = False
VIZ_EVERY = 1
VIZ_SAMPLES = 1
VIZ_SEED = 42
VIZ_RESIDUAL_METRIC = "l1"
FILTER_FULLY_OCEAN_SAMPLES = False

print("Catalog windows:")
for row in list_catalog_windows():
    mark = " <-- selected" if row["window_number"] == FINETUNE_WINDOW else ""
    print(f"  W{row['window_number']:02d} slot={row['window_slot']} {row['export_key']} ({row['label']}){mark}")
print()
print("FINETUNE_WINDOW:", FINETUNE_WINDOW, f"({CATALOG_LABEL})")
print("WINDOW_SLOT:", WINDOW_SLOT, "| export_key:", CATALOG_EXPORT_KEY)
print("DATA_PREFIX:", DATA_PREFIX)
print("INIT_MODEL_PATH:", INIT_MODEL_PATH)
print("RUN_DIR:", RUN_DIR)
print("epochs:", EPOCHS)


Catalog windows:
  W01 slot=0 magicwin_reference (1 reference)
  W02 slot=1 magicwin_coast_slide_n2 (2 coast slide n2)
  W03 slot=2 magicwin_coast_slide_n1 (3 coast slide n1)
  W04 slot=3 magicwin_coast_slide_s1 (4 coast slide s1)
  W05 slot=4 magicwin_coast_slide_s2 (5 coast slide s2)
  W06 slot=5 magicwin_bathy_mid (6 bathy mid)
  W07 slot=6 magicwin_bathy_east (7 bathy east) <-- selected
  W08 slot=7 magicwin_bathy_northeast (8 bathy northeast)
  W09 slot=8 magicwin_rot_0deg (9 rot 0°)
  W10 slot=9 magicwin_rot_23deg (10 rot 23°)
  W11 slot=10 magicwin_rot_45deg (11 rot 45°)
  W12 slot=11 magicwin_rot_68deg (12 rot 68°)

FINETUNE_WINDOW: 7 (7 bathy east)
WINDOW_SLOT: 6 | export_key: magicwin_bathy_east
DATA_PREFIX: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/select_window_12/sample_13_22_uv_dt360
INIT_MODEL_PATH: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_wind

In [42]:
import json
import numpy as np

pool_path = Path(f"{DATA_PREFIX}_sims_pool_k1.npy")
if not pool_path.exists():
    raise FileNotFoundError(
        f"Missing {pool_path}\nRun Shalow/01_export_select_window_pool12.ipynb first."
    )
if not INIT_MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Missing pretrained checkpoint: {INIT_MODEL_PATH}\n"
        f"Train or copy weights from 04_Sh_train_select_window_12.ipynb."
    )

pool_sims = np.load(pool_path, mmap_mode="r")
mask_pool = np.load(f"{DATA_PREFIX}_mask_pool_k1.npy", mmap_mode="r")
n_snap, n_win = pool_sims.shape[:2]
slot_sims = pool_sims[:, WINDOW_SLOT]
slot_mask = mask_pool[:, WINDOW_SLOT]
land_frac = float(slot_mask.mean())

print(f"Pool export: {pool_path.name}")
print(f"  shape: {pool_sims.shape} (snapshots × windows × T × C × H × W)")
print(f"  fine-tune slot {WINDOW_SLOT} (W{FINETUNE_WINDOW:02d}): {slot_sims.shape}")
print(f"  land fraction (slot): {land_frac:.2%}")
print(f"  val split: {int((1-VAL_SPLIT)*100)}% train / {int(VAL_SPLIT*100)}% val ({VAL_SPLIT_MODE}, seed={VAL_SPLIT_SEED})")


Pool export: sample_13_22_uv_dt360_sims_pool_k1.npy
  shape: (3652, 12, 4, 2, 128, 128) (snapshots × windows × T × C × H × W)
  fine-tune slot 6 (W07): (3652, 4, 2, 128, 128)
  land fraction (slot): 0.00%
  val split: 70% train / 30% val (random, seed=42)


In [43]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=EPOCHS,
    batch_size=64,
    lr=2e-4,
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    generator=GENERATOR,
    unet_dim=UNET_DIM,
    unet_dim_mults=UNET_DIM_MULTS,
    unet_dropout=UNET_DROPOUT,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_mask_buffer_px=0,
    filter_fully_ocean_samples=FILTER_FULLY_OCEAN_SAMPLES,
    max_samples=MAX_SAMPLES,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    preview_val_mask=PREVIEW_VAL_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir=str(MODELS_DIR),
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    show_inline_plots=SHOW_INLINE_PLOTS,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
    dynamic_random_crop=DYNAMIC_RANDOM_CROP,
    random_crop_seed=RANDOM_CROP_SEED,
    max_mask_frac=MAX_MASK_FRAC,
    train_crops_per_anchor=TRAIN_CROPS_PER_ANCHOR,
    use_window_pool=USE_WINDOW_POOL,
    slots_per_epoch=TRAIN_SLOTS_PER_EPOCH,
    fixed_window_slot=WINDOW_SLOT,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    use_multi_gpu=USE_MULTI_GPU,
    init_model_path=str(INIT_MODEL_PATH),
    resume_from_best=False,
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))

meta = {
    "finetune_window_number": FINETUNE_WINDOW,
    "finetune_window_slot": WINDOW_SLOT,
    "catalog_export_key": CATALOG_EXPORT_KEY,
    "catalog_label": CATALOG_LABEL,
    "pretrained_run_name": PRETRAINED_RUN_NAME,
    "pretrained_model_path": str(INIT_MODEL_PATH),
    "data_export_key": EXPORT_KEY,
}
(RUN_DIR / "finetune_meta.json").write_text(json.dumps(meta, indent=2))

print(f"Wrote config: {config_path}")
print(f"  fixed catalog slot: {WINDOW_SLOT} (W{FINETUNE_WINDOW:02d} / {CATALOG_EXPORT_KEY})")
print(f"  init weights: {INIT_MODEL_PATH}")
print(f"  epochs: {EPOCHS} | loss: {LOSS_MODE}")


Wrote config: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/fineTuning/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_l1random_finetune_w07/train_config_shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_l1random_finetune_w07.json
  fixed catalog slot: 6 (W07 / magicwin_bathy_east)
  init weights: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6/best_model.pth
  epochs: 100 | loss: l1


In [44]:
import subprocess

LOG_DIR.mkdir(parents=True, exist_ok=True)
log_path = LOG_DIR / f"train_{RUN_NAME}.log"

if DETACHED:
    cmd = [
        sys.executable,
        str(ROOT / "scripts/train_shalow_adv_no.py"),
        "--config",
        str(config_path),
    ]
    if RESUME:
        cmd.append("--resume")
    log_mode = "a" if RESUME and log_path.exists() else "w"
    with open(log_path, log_mode) as log_file:
        proc = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            cwd=ROOT,
        )
    print(f"Started background training PID {proc.pid}")
    print(f"Log: {log_path}")
    print(f"Monitor: tail -f {log_path}")
else:
    print(f"Foreground training — log: {log_path}")
    with open(log_path, "w") as log_file:
        class _Tee:
            def __init__(self, *streams):
                self.streams = streams
            def write(self, data):
                for s in self.streams:
                    s.write(data)
                    s.flush()
            def flush(self):
                for s in self.streams:
                    s.flush()
        _stdout = sys.stdout
        sys.stdout = _Tee(_stdout, log_file)
        try:
            run_adv_no_training(train_config, resume=RESUME)
        finally:
            sys.stdout = _stdout


Started background training PID 3220293
Log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/fineTuning/train_shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_l1random_finetune_w07.log
Monitor: tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/fineTuning/train_shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_l1random_finetune_w07.log


In [45]:
import matplotlib.pyplot as plt
import torch

history_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
state_path = RUN_DIR / f"train_state_{RUN_NAME}.json"
plot_path = RUN_DIR / "viz" / f"training_loss_curve_{RUN_NAME}.png"
log_path = LOG_DIR / f"train_{RUN_NAME}.log"

if not history_path.exists():
    msg = [f"No history yet: {history_path}"]
    if state_path.exists():
        state = json.loads(state_path.read_text())
        msg.append(
            f"Training status: {state.get('status')} | "
            f"last completed epoch: {state.get('last_completed_epoch', 0)}/{state.get('epochs_planned', '?')}"
        )
    if log_path.exists():
        tail = log_path.read_text().splitlines()[-5:]
        msg.append("Recent log lines:")
        msg.extend(f"  {line}" for line in tail)
    msg.append("Re-run this cell after epoch 1 finishes.")
    print("\n".join(msg))
else:
    hist = torch.load(history_path, map_location="cpu", weights_only=False)
    epochs = hist["epoch"]

    if not epochs:
        print("History file exists but is empty. Wait for epoch 1 to finish, then re-run.")
    else:
        recon_label = "speed L1" if LOSS_METRIC == "speed" else "uv L1"
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        ax[0].plot(epochs, hist["train_l1"], label=f"Train {recon_label}")
        ax[0].plot(epochs, hist["val_l1"], label=f"Val {recon_label}")
        ax[0].set_title(f"Reconstruction ({recon_label})")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

        use_adv_no = LOSS_MODE == "adv_no"
        if use_adv_no and any(hist.get("train_perceptual", [])):
            ax[1].plot(epochs, hist["train_perceptual"], label="Train perc")
            ax[1].plot(epochs, hist["val_perceptual"], label="Val perc")
            ax[1].set_title("Perceptual")
            ax[1].legend()
            ax[1].grid(True, alpha=0.3)
        else:
            ax[1].set_title("Perceptual (L1-only run)")
            ax[1].axis("off")

        if use_adv_no and any(hist.get("train_adv", [])):
            ax[2].plot(epochs, hist["train_adv"], label="Train adv")
            ax[2].plot(epochs, hist["train_d"], label="Train D")
            ax[2].set_title("Adversarial terms")
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
        else:
            ax[2].set_title("Adversarial (L1-only run)")
            ax[2].axis("off")

        plt.tight_layout()
        plot_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        if SHOW_INLINE_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print(f"Best val {recon_label}: {hist['best_val_loss']:.6f} at epoch {hist['best_epoch']}")
        print(f"Saved plot: {plot_path}")


No history yet: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/fineTuning/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_l1random_finetune_w07/history/training_history_shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_l1random_finetune_w07.pt
Recent log lines:
Re-run this cell after epoch 1 finishes.
